In [ ]:
import numpy as np
import gymnasium as gym
from typing import Optional

# Custom Environment

In [ ]:
class FoodTruck(gym.Env):
    def __init__(self):
        self.v_demand = [100, 200, 300, 400]
        self.p_demand = [0.3, 0.4, 0.2, 0.1]
        self.capacity = self.v_demand[-1]
        self.days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Weekend']
        self.unit_cost = 4
        self.net_revenue = 7

        # state variables initialization
        self.day = ""
        self.inventory = -1

        # action space setup
        self.actions_list = [0, 100, 200, 300, 400] # number of patties to purchase at the beginning of the day
        self.action_space = gym.spaces.Discrete(len(self.actions_list))

        # observation space setup (fully observable environment: state space and observation space are the same)
        self.state_space = [("Mon", 0)] + [
            (d, i)
            for d in ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Weekend'][1:]
            for i in [0, 100, 200, 300]
        ]
        self.observation_space = gym.spaces.Discrete(len(self.state_space))

    def _get_obs(self):
        return self.state_space.index((self.day, self.inventory))

    def get_next_state_reward(self, state, action, demand):
        day, inventory = state
        result = {}
        result['next_day'] = self.days[self.days.index(day) + 1]
        result['starting_inventory'] = min(self.capacity, inventory + action)
        result['cost'] = self.unit_cost * action
        result['sales'] = min(result['starting_inventory'], demand)
        result['revenue'] = self.net_revenue * result['sales']
        result['next_inventory'] = result['starting_inventory'] - result['sales']
        result['reward'] = result['revenue'] - result['cost']
        return result

    # returns all possible transitions (next state and rewards) for a given state and action pair
    def get_transition_prob(self, state, action):
        next_s_r_prob = {}
        for ix, demand in enumerate(self.v_demand):
            result = self.get_next_state_reward(state, action, demand)
            next_s = (result['next_day'], result['next_inventory'])
            reward = result['reward']
            prob = self.p_demand[ix]

            if (next_s, reward) not in next_s_r_prob:
                next_s_r_prob[next_s, reward] = prob
            else:
                next_s_r_prob[next_s, reward] += prob

        return next_s_r_prob

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self.day = "Mon"
        self.inventory = 0
        observation = self._get_obs()
        info = {} # in this example we are not using info. Instead, you can define your custom '_get_info(self)' method in environment class and call it here.
        return observation, info

    def is_terminal(self, state):
        day, inventory = state
        return day == "Weekend"

    def step(self, action_index):
        # Convert action index back to real action
        action = self.actions_list[action_index]

        demand = np.random.choice(self.v_demand, p=self.p_demand)
        result = self.get_next_state_reward((self.day, self.inventory), action, demand)

        self.day = result['next_day']
        self.inventory = result['next_inventory']

        state = (self.day, self.inventory)
        reward = result['reward']

        terminated = self.is_terminal(state)  # natural termination
        truncated = False                     # no time-limit logic here

        observation = self._get_obs()
        info = {'demand': demand, 'sales': result['sales']}

        return observation, reward, terminated, truncated, info


In [ ]:
foodtruck = FoodTruck()
foodtruck.get_transition_prob(("Tue", 200), 100)

{(('Wed', 200), 300): 0.3,
 (('Wed', 100), 1000): 0.4,
 (('Wed', 0), 1700): 0.30000000000000004}

# Policy Evaluation

In [ ]:
def base_policy(states):
    policy = {}
    for s in states:
        day, inventory = s
        prob_a = {}
        if inventory >= 300:
            prob_a[0] = 1 # no need to replenish
        else: # replenish to 200 or 300 with equal probability
            prob_a[200 - inventory] = 0.5
            prob_a[300 - inventory] = 0.5
        policy[s] = prob_a
    return policy

In [ ]:
foodtruck = FoodTruck()
base_policy(foodtruck.state_space)

{('Mon', 0): {200: 0.5, 300: 0.5},
 ('Tue', 0): {200: 0.5, 300: 0.5},
 ('Tue', 100): {100: 0.5, 200: 0.5},
 ('Tue', 200): {0: 0.5, 100: 0.5},
 ('Tue', 300): {0: 1},
 ('Wed', 0): {200: 0.5, 300: 0.5},
 ('Wed', 100): {100: 0.5, 200: 0.5},
 ('Wed', 200): {0: 0.5, 100: 0.5},
 ('Wed', 300): {0: 1},
 ('Thu', 0): {200: 0.5, 300: 0.5},
 ('Thu', 100): {100: 0.5, 200: 0.5},
 ('Thu', 200): {0: 0.5, 100: 0.5},
 ('Thu', 300): {0: 1},
 ('Fri', 0): {200: 0.5, 300: 0.5},
 ('Fri', 100): {100: 0.5, 200: 0.5},
 ('Fri', 200): {0: 0.5, 100: 0.5},
 ('Fri', 300): {0: 1},
 ('Weekend', 0): {200: 0.5, 300: 0.5},
 ('Weekend', 100): {100: 0.5, 200: 0.5},
 ('Weekend', 200): {0: 0.5, 100: 0.5},
 ('Weekend', 300): {0: 1}}

In [ ]:
def expected_update(env, v, s, prob_a, gamma):
    expected_value = 0
    for a in prob_a:
        prob_next_s_r = env.get_transition_prob(s, a)
        for next_s, r in prob_next_s_r:
            expected_value += prob_a[a] * prob_next_s_r[next_s, r] * (r + gamma * v[next_s])
    return expected_value

Iterative Policy Evaluation

In [ ]:
def iterative_policy_evaluation(env, policy, max_iter=100, v = None, eps=0.1, gamma=1):
    if not v:
        v = {s: 0 for s in env.state_space}
    k = 0
    while True:
        max_delta = 0
        for s in v:
            if not env.is_terminal(s):
                v_old = v[s]
                prob_a = policy[s]
                v[s] = expected_update(env, v, s, prob_a, gamma)
                max_delta = max(max_delta, abs(v[s] - v_old))
        k += 1
        if max_delta < eps:
            print("Converged in", k, "iterations.")
            break
        elif k == max_iter:
            print("Terminating after", k, "iterations.")
            break
    return v

In [ ]:
foodtruck = FoodTruck()
policy = base_policy(foodtruck.state_space)

In [ ]:
v = iterative_policy_evaluation(foodtruck, policy)
print("Expected weekly profit:", v["Mon", 0])

Converged in 6 iterations.
Expected weekly profit: 2515.0


In [ ]:
print("The state values:")
v

The state values:


{('Mon', 0): 2515.0,
 ('Tue', 0): 1960.0,
 ('Tue', 100): 2360.0,
 ('Tue', 200): 2760.0,
 ('Tue', 300): 3205.0,
 ('Wed', 0): 1405.0,
 ('Wed', 100): 1805.0,
 ('Wed', 200): 2205.0,
 ('Wed', 300): 2650.0,
 ('Thu', 0): 850.0000000000001,
 ('Thu', 100): 1250.0,
 ('Thu', 200): 1650.0,
 ('Thu', 300): 2095.0,
 ('Fri', 0): 295.00000000000006,
 ('Fri', 100): 695.0000000000001,
 ('Fri', 200): 1095.0,
 ('Fri', 300): 1400.0,
 ('Weekend', 0): 0,
 ('Weekend', 100): 0,
 ('Weekend', 200): 0,
 ('Weekend', 300): 0}

Simulation with Gym Environment

In [ ]:
def choose_action(state, policy):
    prob_a = policy[state]
    action = np.random.choice(a=list(prob_a.keys()),
                              p=list(prob_a.values()))
    return action

def simulate_policy(policy, n_episodes):
    np.random.seed(0)
    foodtruck = FoodTruck()
    rewards = []
    for i_episode in range(n_episodes):
        observation, info = foodtruck.reset()
        state = foodtruck.state_space[observation]
        episode_over = False
        ep_reward = 0
        while not episode_over:
            action = choose_action(state, policy)
            action_index = foodtruck.actions_list.index(action)
            observation, reward, terminated, truncated, info = foodtruck.step(action_index)
            state = foodtruck.state_space[observation]
            ep_reward += reward
            episode_over = terminated or truncated
        rewards.append(ep_reward)
    print("Expected weekly profit:", np.mean(rewards))

In [ ]:
simulate_policy(policy, 1000)

Expected weekly profit: 2518.1


# Policy Iteration

In [ ]:
# How does a random policy (deterministic) look like?
np.random.seed(1)
policy = {s: {np.random.choice(foodtruck.actions_list): 1} for s in foodtruck.state_space}
print(policy)

{('Mon', 0): {np.int64(300): 1}, ('Tue', 0): {np.int64(400): 1}, ('Tue', 100): {np.int64(0): 1}, ('Tue', 200): {np.int64(100): 1}, ('Tue', 300): {np.int64(300): 1}, ('Wed', 0): {np.int64(0): 1}, ('Wed', 100): {np.int64(0): 1}, ('Wed', 200): {np.int64(100): 1}, ('Wed', 300): {np.int64(400): 1}, ('Thu', 0): {np.int64(400): 1}, ('Thu', 100): {np.int64(100): 1}, ('Thu', 200): {np.int64(200): 1}, ('Thu', 300): {np.int64(400): 1}, ('Fri', 0): {np.int64(200): 1}, ('Fri', 100): {np.int64(400): 1}, ('Fri', 200): {np.int64(300): 1}, ('Fri', 300): {np.int64(400): 1}, ('Weekend', 0): {np.int64(200): 1}, ('Weekend', 100): {np.int64(400): 1}, ('Weekend', 200): {np.int64(200): 1}, ('Weekend', 300): {np.int64(400): 1}}


In [ ]:
# searches for the action that gives the maximum q-value for a given state
def policy_improvement(env, v, s, actions, gamma):
    prob_a = {}
    if not env.is_terminal(s):
        max_q = -np.inf
        best_a = None
        for a in actions:
            q_sa = expected_update(env, v, s, {a: 1}, gamma) # {a: 1} means 100% prob. of taking action a (deterministic approach)
            if q_sa >= max_q:
                max_q = q_sa
                best_a = a
        prob_a[best_a] = 1
    else:
        max_q = 0 #zero q value is returned for terminal states (weekend)
    return prob_a, max_q

def policy_iteration(env,  eps=0.1, gamma=1):
    np.random.seed(1)
    states = env.state_space
    actions = env.actions_list # true actions, not indices in action_space
    policy = {s: {np.random.choice(actions): 1}
             for s in states}
    v = {s: 0 for s in states} # init all v values to zero
    while True:
        v = iterative_policy_evaluation(env, policy, v=v, eps=eps, gamma=gamma)
        old_policy = policy
        policy = {}
        for s in states:
            policy[s], _ = policy_improvement(env, v, s, actions, gamma)
        if old_policy == policy:
            break
    print("Optimal policy found!")
    return policy, v

In [ ]:
policy, v = policy_iteration(foodtruck)
print("Expected weekly profit:", v["Mon", 0])

Converged in 6 iterations.
Converged in 6 iterations.
Converged in 5 iterations.
Optimal policy found!
Expected weekly profit: 2880.0


In [ ]:
policy

{('Mon', 0): {400: 1},
 ('Tue', 0): {400: 1},
 ('Tue', 100): {300: 1},
 ('Tue', 200): {200: 1},
 ('Tue', 300): {100: 1},
 ('Wed', 0): {400: 1},
 ('Wed', 100): {300: 1},
 ('Wed', 200): {200: 1},
 ('Wed', 300): {100: 1},
 ('Thu', 0): {300: 1},
 ('Thu', 100): {200: 1},
 ('Thu', 200): {100: 1},
 ('Thu', 300): {0: 1},
 ('Fri', 0): {200: 1},
 ('Fri', 100): {100: 1},
 ('Fri', 200): {0: 1},
 ('Fri', 300): {0: 1},
 ('Weekend', 0): {},
 ('Weekend', 100): {},
 ('Weekend', 200): {},
 ('Weekend', 300): {}}

# Value Iteration

In [ ]:
def value_iteration(env, max_iter=100, eps=0.1, gamma=1):
    states = env.state_space
    actions = env.actions_list
    v = {s: 0 for s in states}
    policy = {}
    k = 0
    while True:
        max_delta = 0
        for s in states:
            old_v = v[s]
            policy[s], v[s] = policy_improvement(env,
                                                 v,
                                                 s,
                                                 actions,
                                                 gamma)
            max_delta = max(max_delta, abs(v[s] - old_v))
        k += 1
        if max_delta < eps:
            print("Converged in", k, "iterations.")
            break
        elif k == max_iter:
            print("Terminating after", k, "iterations.")
            break
    return policy, v

In [ ]:
policy, v = value_iteration(foodtruck)
print("Expected weekly profit:", v["Mon", 0])

Converged in 6 iterations.
Expected weekly profit: 2880.0


In [ ]:
policy

{('Mon', 0): {400: 1},
 ('Tue', 0): {400: 1},
 ('Tue', 100): {300: 1},
 ('Tue', 200): {200: 1},
 ('Tue', 300): {100: 1},
 ('Wed', 0): {400: 1},
 ('Wed', 100): {300: 1},
 ('Wed', 200): {200: 1},
 ('Wed', 300): {100: 1},
 ('Thu', 0): {300: 1},
 ('Thu', 100): {200: 1},
 ('Thu', 200): {100: 1},
 ('Thu', 300): {0: 1},
 ('Fri', 0): {200: 1},
 ('Fri', 100): {100: 1},
 ('Fri', 200): {0: 1},
 ('Fri', 300): {0: 1},
 ('Weekend', 0): {},
 ('Weekend', 100): {},
 ('Weekend', 200): {},
 ('Weekend', 300): {}}

# Monte Carlo (MC) Methods

## MC Prediction

In [ ]:
def first_visit_return(returns, trajectory, gamma):
    G = 0
    T = len(trajectory) - 1
    for t, sar in enumerate(reversed(trajectory)):
        s, a, r = sar
        G = r + gamma * G
        first_visit = True
        for j in range(T - t):
            if s == trajectory[j][0]:
                first_visit = False
        if first_visit: # in the trajectory
            if s in returns:
                returns[s].append(G)
            else:
                returns[s] = [G]
    return returns

In [ ]:
# simulates the environment for a single episode with a given policy and returns a trajectory
def get_trajectory(env, policy):
    trajectory = []
    observation, info = env.reset()
    state = env.state_space[observation] # convert gymnasium observation to state
    episode_over = False
    sar = [state]
    while not episode_over:
        action = choose_action(state, policy)
        action_index = foodtruck.actions_list.index(action) # action index is needed for step function (it uses discrete action space 0,1,...)
        observation, reward, terminated, truncated, info = env.step(action_index)
        state = env.state_space[observation]
        sar.append(action)
        sar.append(reward)
        trajectory.append(sar)
        sar = [state]
        episode_over = terminated or truncated
    return trajectory

In [ ]:
def first_visit_mc(env, policy, gamma, n_trajectories):
    np.random.seed(0)
    returns = {}
    v = {}
    for i in range(n_trajectories):
        trajectory = get_trajectory(env, policy)
        returns = first_visit_return(returns,
                                     trajectory,
                                     gamma)
    for s in env.state_space:
        if s in returns:
            v[s] = np.round(np.mean(returns[s]), 1)
    return v

In [ ]:
foodtruck = FoodTruck()
policy = base_policy(foodtruck.state_space)

In [ ]:
v_est = first_visit_mc(foodtruck, policy, 1, 10000)
v_est

{('Mon', 0): np.float64(2515.9),
 ('Tue', 0): np.float64(1959.1),
 ('Tue', 100): np.float64(2362.2),
 ('Tue', 200): np.float64(2765.2),
 ('Wed', 0): np.float64(1411.3),
 ('Wed', 100): np.float64(1804.2),
 ('Wed', 200): np.float64(2198.9),
 ('Thu', 0): np.float64(852.9),
 ('Thu', 100): np.float64(1265.4),
 ('Thu', 200): np.float64(1644.4),
 ('Fri', 0): np.float64(301.1),
 ('Fri', 100): np.float64(696.5),
 ('Fri', 200): np.float64(1097.2)}

In [ ]:
v_true = iterative_policy_evaluation(foodtruck, policy)
v_true

Converged in 6 iterations.


{('Mon', 0): 2515.0,
 ('Tue', 0): 1960.0,
 ('Tue', 100): 2360.0,
 ('Tue', 200): 2760.0,
 ('Tue', 300): 3205.0,
 ('Wed', 0): 1405.0,
 ('Wed', 100): 1805.0,
 ('Wed', 200): 2205.0,
 ('Wed', 300): 2650.0,
 ('Thu', 0): 850.0000000000001,
 ('Thu', 100): 1250.0,
 ('Thu', 200): 1650.0,
 ('Thu', 300): 2095.0,
 ('Fri', 0): 295.00000000000006,
 ('Fri', 100): 695.0000000000001,
 ('Fri', 200): 1095.0,
 ('Fri', 300): 1400.0,
 ('Weekend', 0): 0,
 ('Weekend', 100): 0,
 ('Weekend', 200): 0,
 ('Weekend', 300): 0}

## On-policy Monte Carlo Control

In [ ]:
def get_eps_greedy(actions, eps, a_best):
    prob_a = {}
    n_a = len(actions)
    for a in actions:
        if a == a_best:
            prob_a[a] = 1 - eps + eps/n_a
        else:
            prob_a[a] = eps/n_a
    return prob_a

In [ ]:
def get_random_policy(states, actions):
    policy = {}
    n_a = len(actions)
    for s in states:
        policy[s] = {a: 1/n_a for a in actions}
    return policy

In [ ]:
import operator

def on_policy_first_visit_mc(env, n_iter, eps, gamma):
    np.random.seed(0)
    states =  env.state_space
    actions = env.actions_list
    policy =  get_random_policy(states, actions)
    Q = {s: {a: 0 for a in actions} for s in states}
    Q_n = {s: {a: 0 for a in actions} for s in states}
    for i in range(n_iter):
        if i % 10000 == 0:
            print("Iteration:", i)
        trajectory = get_trajectory(env, policy)
        G = 0
        T = len(trajectory) - 1
        for t, sar in enumerate(reversed(trajectory)):
            s, a, r = sar
            G = r + gamma * G
            first_visit = True
            for j in range(T - t):
                s_j = trajectory[j][0]
                a_j = trajectory[j][1]
                if (s, a) == (s_j, a_j):
                    first_visit = False
            if first_visit:
                Q[s][a] = Q_n[s][a] * Q[s][a] + G
                Q_n[s][a] += 1
                Q[s][a] /= Q_n[s][a]
                a_best = max(Q[s].items(), key=operator.itemgetter(1))[0]
                policy[s] = get_eps_greedy(actions, eps, a_best)
    return policy, Q, Q_n

In [ ]:
policy, Q, Q_n = on_policy_first_visit_mc(foodtruck, 300000, 0.05, 1)

Iteration: 0
Iteration: 10000
Iteration: 20000
Iteration: 30000
Iteration: 40000
Iteration: 50000
Iteration: 60000
Iteration: 70000
Iteration: 80000
Iteration: 90000
Iteration: 100000
Iteration: 110000
Iteration: 120000
Iteration: 130000
Iteration: 140000
Iteration: 150000
Iteration: 160000
Iteration: 170000
Iteration: 180000
Iteration: 190000
Iteration: 200000
Iteration: 210000
Iteration: 220000
Iteration: 230000
Iteration: 240000
Iteration: 250000
Iteration: 260000
Iteration: 270000
Iteration: 280000
Iteration: 290000


In [ ]:
policy

{('Mon', 0): {0: 0.01, 100: 0.01, 200: 0.01, 300: 0.01, 400: 0.96},
 ('Tue', 0): {0: 0.01, 100: 0.01, 200: 0.01, 300: 0.01, 400: 0.96},
 ('Tue', 100): {0: 0.01, 100: 0.01, 200: 0.01, 300: 0.96, 400: 0.01},
 ('Tue', 200): {0: 0.01, 100: 0.01, 200: 0.96, 300: 0.01, 400: 0.01},
 ('Tue', 300): {0: 0.01, 100: 0.96, 200: 0.01, 300: 0.01, 400: 0.01},
 ('Wed', 0): {0: 0.01, 100: 0.01, 200: 0.01, 300: 0.01, 400: 0.96},
 ('Wed', 100): {0: 0.01, 100: 0.01, 200: 0.01, 300: 0.96, 400: 0.01},
 ('Wed', 200): {0: 0.01, 100: 0.01, 200: 0.96, 300: 0.01, 400: 0.01},
 ('Wed', 300): {0: 0.01, 100: 0.96, 200: 0.01, 300: 0.01, 400: 0.01},
 ('Thu', 0): {0: 0.01, 100: 0.01, 200: 0.01, 300: 0.96, 400: 0.01},
 ('Thu', 100): {0: 0.01, 100: 0.01, 200: 0.96, 300: 0.01, 400: 0.01},
 ('Thu', 200): {0: 0.01, 100: 0.96, 200: 0.01, 300: 0.01, 400: 0.01},
 ('Thu', 300): {0: 0.96, 100: 0.01, 200: 0.01, 300: 0.01, 400: 0.01},
 ('Fri', 0): {0: 0.01, 100: 0.01, 200: 0.96, 300: 0.01, 400: 0.01},
 ('Fri', 100): {0: 0.01, 100: 

In [ ]:
Q

{('Mon', 0): {0: np.float64(2162.733333333329),
  100: np.float64(2468.4210526315796),
  200: np.float64(2668.7695190505888),
  300: np.float64(2739.300098231826),
  400: np.float64(2809.1632287569414)},
 ('Tue', 0): {0: np.float64(1539.1011235955057),
  100: np.float64(1857.630979498861),
  200: np.float64(2018.3222958057395),
  300: np.float64(2101.97486535009),
  400: np.float64(2181.249139237035)},
 ('Tue', 100): {0: np.float64(2243.7967115097176),
  100: np.float64(2410.7182940516295),
  200: np.float64(2537.853107344635),
  300: np.float64(2587.222441722628),
  400: np.float64(2170.4049844236765)},
 ('Tue', 200): {0: np.float64(2828.295819935689),
  100: np.float64(2953.6330631123433),
  200: np.float64(2996.437255166801),
  300: np.float64(2623.82297551789),
  400: np.float64(2224.710080285464)},
 ('Tue', 300): {0: np.float64(3383.880037488284),
  100: np.float64(3395.720002238628),
  200: np.float64(2939.4218134034168),
  300: np.float64(2572.2506393861877),
  400: np.float64(2

## Off-policy Monte Carlo Control

In [ ]:
def off_policy_mc(env, n_iter, eps, gamma):
    np.random.seed(0)
    states =  env.state_space
    actions = env.actions_list
    Q = {s: {a: 0 for a in actions} for s in states}
    C = {s: {a: 0 for a in actions} for s in states}
    target_policy = {}
    behavior_policy = get_random_policy(states,
                                        actions)
    for i in range(n_iter):
        if i % 10000 == 0:
            print("Iteration:", i)
        trajectory = get_trajectory(env,
                                    behavior_policy)
        G = 0
        W = 1
        T = len(trajectory) - 1
        for t, sar in enumerate(reversed(trajectory)):
            s, a, r = sar
            G = r + gamma * G
            C[s][a] += W
            Q[s][a] += (W/C[s][a]) * (G - Q[s][a])
            a_best = max(Q[s].items(),
                         key=operator.itemgetter(1))[0]
            target_policy[s] = a_best
            behavior_policy[s] = get_eps_greedy(actions,
                                                eps,
                                                a_best)
            if a != target_policy[s]:
                break
            W = W / behavior_policy[s][a]
    target_policy = {s: target_policy[s] for s in states
                                   if s in target_policy}
    return target_policy, Q

In [ ]:
policy, Q = off_policy_mc(foodtruck, 300000, 0.05, 1)

Iteration: 0
Iteration: 10000
Iteration: 20000
Iteration: 30000
Iteration: 40000
Iteration: 50000
Iteration: 60000
Iteration: 70000
Iteration: 80000
Iteration: 90000
Iteration: 100000
Iteration: 110000
Iteration: 120000
Iteration: 130000
Iteration: 140000
Iteration: 150000
Iteration: 160000
Iteration: 170000
Iteration: 180000
Iteration: 190000
Iteration: 200000
Iteration: 210000
Iteration: 220000
Iteration: 230000
Iteration: 240000
Iteration: 250000
Iteration: 260000
Iteration: 270000
Iteration: 280000
Iteration: 290000


In [ ]:
policy

{('Mon', 0): 400,
 ('Tue', 0): 400,
 ('Tue', 100): 300,
 ('Tue', 200): 200,
 ('Tue', 300): 100,
 ('Wed', 0): 400,
 ('Wed', 100): 300,
 ('Wed', 200): 200,
 ('Wed', 300): 100,
 ('Thu', 0): 300,
 ('Thu', 100): 200,
 ('Thu', 200): 100,
 ('Thu', 300): 0,
 ('Fri', 0): 200,
 ('Fri', 100): 100,
 ('Fri', 200): 0,
 ('Fri', 300): 0}

In [ ]:
Q

{('Mon', 0): {0: np.float64(2232.674050632915),
  100: np.float64(2539.364696421396),
  200: np.float64(2725.681570338065),
  300: np.float64(2822.8136882129284),
  400: np.float64(2878.458190025779)},
 ('Tue', 0): {0: np.float64(1594.8051948051952),
  100: np.float64(1928.976034858388),
  200: np.float64(2067.4576271186465),
  300: np.float64(2207.8512396694205),
  400: np.float64(2239.8886329583893)},
 ('Tue', 100): {0: np.float64(2318.9435336976317),
  100: np.float64(2536.8012422360302),
  200: np.float64(2549.486301369862),
  300: np.float64(2650.193090274893),
  400: np.float64(2256.120527306967)},
 ('Tue', 200): {0: np.float64(2922.175290390706),
  100: np.float64(3012.8990770161868),
  200: np.float64(3052.4769607403373),
  300: np.float64(2689.515219842163),
  400: np.float64(2293.305439330548)},
 ('Tue', 300): {0: np.float64(3420.032031538755),
  100: np.float64(3453.749726573689),
  200: np.float64(3014.1210374639763),
  300: np.float64(2635.802469135803),
  400: np.float64(

# TD Learning

## TD Prediction

In [ ]:
def one_step_td_prediction(env, policy, gamma, alpha, n_iter):
    np.random.seed(0)
    states = env.state_space
    v = {s: 0 for s in states}
    observation, info = env.reset()
    s = env.state_space[observation]
    for i in range(n_iter):
        a = choose_action(s, policy)
        action_index = env.actions_list.index(a)
        observation, reward, terminated, truncated, info = env.step(action_index)
        s_next = env.state_space[observation]
        v[s] += alpha * (reward + gamma * v[s_next] - v[s])
        if terminated or truncated:
            observation, info = env.reset()
            s = env.state_space[observation]
        else:
            s = s_next
    return v

In [ ]:
policy = base_policy(foodtruck.state_space)
v = one_step_td_prediction(foodtruck, policy, 1, 0.01, 100000)
v

{('Mon', 0): np.float64(2506.576417395407),
 ('Tue', 0): np.float64(1956.077876400167),
 ('Tue', 100): np.float64(2368.7400039407535),
 ('Tue', 200): np.float64(2767.5069659225423),
 ('Tue', 300): 0,
 ('Wed', 0): np.float64(1413.0055559001296),
 ('Wed', 100): np.float64(1813.546186490315),
 ('Wed', 200): np.float64(2200.8873259700867),
 ('Wed', 300): 0,
 ('Thu', 0): np.float64(828.2915189850011),
 ('Thu', 100): np.float64(1280.424626614422),
 ('Thu', 200): np.float64(1675.8661846955831),
 ('Thu', 300): 0,
 ('Fri', 0): np.float64(345.52991944823583),
 ('Fri', 100): np.float64(677.4358179389413),
 ('Fri', 200): np.float64(1094.8263154150825),
 ('Fri', 300): 0,
 ('Weekend', 0): 0,
 ('Weekend', 100): 0,
 ('Weekend', 200): 0,
 ('Weekend', 300): 0}

### SARSA

In [ ]:
def sarsa(env, gamma, eps, alpha, n_iter):
    np.random.seed(0)
    states = env.state_space
    actions = env.actions_list
    Q = {s: {a: 0 for a in actions} for s in states}
    policy = get_random_policy(states, actions)
    observation, info = env.reset()
    s = env.state_space[observation]
    a = choose_action(s, policy)
    action_index = env.actions_list.index(a)
    for i in range(n_iter):
        if i % 100000 == 0:
            print("Iteration:", i)
        observation, reward, terminated, truncated, info = env.step(action_index)
        s_next = env.state_space[observation]
        a_best = max(Q[s_next].items(), key=operator.itemgetter(1))[0]
        policy[s_next] = get_eps_greedy(actions, eps, a_best)
        a_next = choose_action(s_next, policy)
        Q[s][a] += alpha * (reward + gamma * Q[s_next][a_next] - Q[s][a])
        if terminated or truncated:
            observation, info = env.reset()
            s = env.state_space[observation]
            a_best = max(Q[s].items(), key=operator.itemgetter(1))[0]
            policy[s] = get_eps_greedy(actions, eps, a_best)
            a = choose_action(s, policy)
            action_index = env.actions_list.index(a)
        else:
            s = s_next
            a = a_next
    return policy, Q

In [ ]:
policy, Q = sarsa(foodtruck, 1, 0.1, 0.01, 1000000)

Iteration: 0
Iteration: 100000
Iteration: 200000
Iteration: 300000
Iteration: 400000
Iteration: 500000
Iteration: 600000
Iteration: 700000
Iteration: 800000
Iteration: 900000


In [ ]:
policy

{('Mon', 0): {0: 0.02, 100: 0.02, 200: 0.92, 300: 0.02, 400: 0.02},
 ('Tue', 0): {0: 0.02, 100: 0.92, 200: 0.02, 300: 0.02, 400: 0.02},
 ('Tue', 100): {0: 0.92, 100: 0.02, 200: 0.02, 300: 0.02, 400: 0.02},
 ('Tue', 200): {0: 0.02, 100: 0.02, 200: 0.92, 300: 0.02, 400: 0.02},
 ('Tue', 300): {0: 0.02, 100: 0.92, 200: 0.02, 300: 0.02, 400: 0.02},
 ('Wed', 0): {0: 0.02, 100: 0.02, 200: 0.02, 300: 0.92, 400: 0.02},
 ('Wed', 100): {0: 0.02, 100: 0.02, 200: 0.02, 300: 0.02, 400: 0.92},
 ('Wed', 200): {0: 0.02, 100: 0.02, 200: 0.02, 300: 0.92, 400: 0.02},
 ('Wed', 300): {0: 0.02, 100: 0.02, 200: 0.02, 300: 0.02, 400: 0.92},
 ('Thu', 0): {0: 0.02, 100: 0.02, 200: 0.92, 300: 0.02, 400: 0.02},
 ('Thu', 100): {0: 0.92, 100: 0.02, 200: 0.02, 300: 0.02, 400: 0.02},
 ('Thu', 200): {0: 0.92, 100: 0.02, 200: 0.02, 300: 0.02, 400: 0.02},
 ('Thu', 300): {0: 0.02, 100: 0.02, 200: 0.92, 300: 0.02, 400: 0.02},
 ('Fri', 0): {0: 0.02, 100: 0.02, 200: 0.02, 300: 0.92, 400: 0.02},
 ('Fri', 100): {0: 0.02, 100: 

In [ ]:
Q

{('Mon', 0): {0: np.float64(1730.7536269649243),
  100: np.float64(2028.7830247689033),
  200: np.float64(2205.319766778804),
  300: np.float64(1906.2266578099532),
  400: np.float64(1390.7646561858276)},
 ('Tue', 0): {0: np.float64(1706.8243883539317),
  100: np.float64(1721.2938163114434),
  200: np.float64(1682.2392278025006),
  300: np.float64(1704.48416635491),
  400: np.float64(1706.3192187284499)},
 ('Tue', 100): {0: np.float64(2090.341651668151),
  100: np.float64(2045.4424429078194),
  200: np.float64(2053.320074701719),
  300: np.float64(2040.107009646765),
  400: np.float64(2049.3931633947013)},
 ('Tue', 200): {0: np.float64(588.9776596990399),
  100: np.float64(687.4051925344852),
  200: np.float64(1406.1632328366338),
  300: np.float64(742.8268516981524),
  400: np.float64(651.3725645437564)},
 ('Tue', 300): {0: np.float64(221.39679852971267),
  100: np.float64(1332.819594735351),
  200: np.float64(227.2645327886909),
  300: np.float64(242.2358995553836),
  400: np.float64

### Q-learning

In [ ]:
def q_learning(env, gamma, eps, alpha, n_iter):
    np.random.seed(0)
    states =  env.state_space
    actions = env.actions_list
    Q = {s: {a: 0 for a in actions} for s in states}
    policy = get_random_policy(states, actions)
    observation, info = env.reset()
    s = env.state_space[observation]
    for i in range(n_iter):
        if i % 100000 == 0:
            print("Iteration:", i)
        a_best = max(Q[s].items(), key=operator.itemgetter(1))[0]
        policy[s] = get_eps_greedy(actions, eps, a_best)
        a = choose_action(s, policy)
        action_index = env.actions_list.index(a)
        observation, reward, terminated, truncated, info = env.step(action_index)
        s_next = env.state_space[observation]
        Q[s][a] += alpha * (reward + gamma * max(Q[s_next].values()) - Q[s][a])
        if terminated or truncated:
            observation, info = env.reset()
            s = env.state_space[observation]
        else:
            s = s_next
    policy = {s: {max(policy[s].items(), key=operator.itemgetter(1))[0]: 1}for s in states}
    return policy, Q

In [ ]:
policy, Q = q_learning(foodtruck, 1, 0.1, 0.01, 1000000)
policy

Iteration: 0
Iteration: 100000
Iteration: 200000
Iteration: 300000
Iteration: 400000
Iteration: 500000
Iteration: 600000
Iteration: 700000
Iteration: 800000
Iteration: 900000


{('Mon', 0): {400: 1},
 ('Tue', 0): {400: 1},
 ('Tue', 100): {300: 1},
 ('Tue', 200): {200: 1},
 ('Tue', 300): {100: 1},
 ('Wed', 0): {400: 1},
 ('Wed', 100): {300: 1},
 ('Wed', 200): {200: 1},
 ('Wed', 300): {100: 1},
 ('Thu', 0): {300: 1},
 ('Thu', 100): {200: 1},
 ('Thu', 200): {100: 1},
 ('Thu', 300): {0: 1},
 ('Fri', 0): {200: 1},
 ('Fri', 100): {100: 1},
 ('Fri', 200): {0: 1},
 ('Fri', 300): {0: 1},
 ('Weekend', 0): {0: 1},
 ('Weekend', 100): {0: 1},
 ('Weekend', 200): {0: 1},
 ('Weekend', 300): {0: 1}}